<a href="https://colab.research.google.com/github/reemacodess/MATH319-GroupProject/blob/main/Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import sqlite3, hashlib, hmac, secrets, os
from datetime import datetime, timezone
import ipywidgets as widgets
from IPython.display import display, clear_output

DB_PATH = 'voting.db'  # Colab temporary storage; export if needed
CANDIDATES = {'A': 'Candidate A', 'B': 'Candidate B', 'C': 'Candidate C'}

def hash_password(password, salt=None):
    salt = salt or os.urandom(16)
    derived = hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), salt, 200_000)
    return salt.hex(), derived.hex()

def verify_password(password, salt_hex, expected_hex):
    _, result = hash_password(password, bytes.fromhex(salt_hex))
    return hmac.compare_digest(result, expected_hex)

def connection():
    con = sqlite3.connect(DB_PATH)
    con.execute('PRAGMA foreign_keys = ON')
    return con

def initialize_students():
    with connection() as con:
        con.execute("""CREATE TABLE IF NOT EXISTS students (
            student_id TEXT PRIMARY KEY,
            password_salt TEXT NOT NULL,
            password_hash TEXT NOT NULL,
            eligible INTEGER NOT NULL CHECK(eligible IN (0,1))
        )""")

def add_demo_student(student_id, password, eligible):
    salt, digest = hash_password(password)
    with connection() as con:
        con.execute('INSERT OR IGNORE INTO students VALUES (?, ?, ?, ?)',
                    (student_id, salt, digest, int(eligible)))

def authenticate(student_id, password):
    with connection() as con:
        row = con.execute('SELECT password_salt, password_hash, eligible FROM students WHERE student_id=?',
                          (student_id,)).fetchone()
    if row is None or not verify_password(password, row[0], row[1]):
        return False, 'Invalid student ID or password.'
    if not row[2]:
        return False, 'This student is not eligible to vote.'
    return True, 'Login successful. Student is eligible.'

initialize_students()
add_demo_student('student01', 'DemoPass01!', True)
add_demo_student('student02', 'DemoPass02!', True)
add_demo_student('student03', 'DemoPass03!', False)
print("Database and demo student accounts are ready.")

Database and demo student accounts are ready.


In [12]:
#convert the original token into a SHA-256 hash
def hash_token(token):
    return hashlib.sha256(token.encode('utf-8')).hexdigest()


#tokens table
def initialize_tokens():
    con = connection()

    try:
        con.execute("""
            CREATE TABLE IF NOT EXISTS tokens (
                student_id TEXT PRIMARY KEY,
                token_hash TEXT NOT NULL UNIQUE,
                is_used INTEGER NOT NULL DEFAULT 0
                    CHECK(is_used IN (0, 1)),
                issued_at TEXT NOT NULL,
                FOREIGN KEY (student_id)
                    REFERENCES students(student_id)
            )
        """)

        con.commit()
        print("Tokens table is ready.")

    finally:
        con.close()


#create the votes table
def initialize_votes():
    con = connection()

    try:
        con.execute("""
            CREATE TABLE IF NOT EXISTS votes (
                vote_id INTEGER PRIMARY KEY AUTOINCREMENT,
                encrypted_vote TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
        """)

        con.commit()
        print("Votes table is ready.")

    finally:
        con.close()


#one random token to each eligible student
def issue_token(student_id):
    con = connection()

    try:
        con.execute("BEGIN IMMEDIATE")

        student = con.execute(
            "SELECT eligible FROM students WHERE student_id = ?",
            (student_id,)
        ).fetchone()

        if student is None or student[0] != 1:
            con.rollback()
            return None, "Student is not eligible to receive a token."

        existing = con.execute(
            "SELECT student_id FROM tokens WHERE student_id = ?",
            (student_id,)
        ).fetchone()

        if existing is not None:
            con.rollback()
            return None, "A token has already been issued to this student."

        #genrate a secure random token
        token = secrets.token_urlsafe(32)

        #store only its hash
        token_digest = hash_token(token)

        con.execute("""
            INSERT INTO tokens
            (student_id, token_hash, is_used, issued_at)
            VALUES (?, ?, 0, ?)
        """, (
            student_id,
            token_digest,
            datetime.now(timezone.utc).isoformat()
        ))

        con.commit()

        #return the original token only once
        return token, "Token issued successfully. Save it safely."

    except Exception:
        con.rollback()
        return None, "Could not issue token."

    finally:
        con.close()


#verfi that the token belongs to the student and is unused
def verify_token(student_id, token):
    if not token:
        return False, "Token is required."

    con = connection()

    try:
        row = con.execute("""
            SELECT token_hash, is_used
            FROM tokens
            WHERE student_id = ?
        """, (student_id,)).fetchone()

        if row is None:
            return False, "No token was found for this student."

        expected_hash, is_used = row

        if not hmac.compare_digest(hash_token(token), expected_hash):
            return False, "Invalid token."

        if is_used == 1:
            return False, "This token has already been used."

        return True, "Token is valid."

    finally:
        con.close()



#replace this with Student 3's real AES encryption function.
def demo_encrypt_vote(candidate_id):
    return "DEMO_ONLY_" + candidate_id


#validate, consume the token, and store the vote atomically
def cast_vote(student_id, token, candidate_id,
              encrypt_vote, deadline):

    if candidate_id not in CANDIDATES:
        return False, "Invalid candidate."

    #check the voting deadline
    now = datetime.now(timezone.utc)

    if deadline is None or deadline.tzinfo is None:
        return False, "Voting deadline is not configured correctly."

    if now >= deadline:
        return False, "Voting is closed."

    #encrypt the vote before opening the database transaction
    try:
        encrypted_vote = encrypt_vote(candidate_id)
    except Exception:
        return False, "Vote encryption failed."

    con = connection()

    try:
        #lock the database for this transaction
        con.execute("BEGIN IMMEDIATE")

        row = con.execute("""
            SELECT token_hash, is_used
            FROM tokens
            WHERE student_id = ?
        """, (student_id,)).fetchone()

        if row is None:
            con.rollback()
            return False, "No token was found."

        expected_hash, is_used = row

        if not hmac.compare_digest(hash_token(token), expected_hash):
            con.rollback()
            return False, "Invalid token."

        if is_used == 1:
            con.rollback()
            return False, "This token has already been used."

        #mark the token as used only if it is still unused
        result = con.execute("""
            UPDATE tokens
            SET is_used = 1
            WHERE student_id = ? AND is_used = 0
        """, (student_id,))

        if result.rowcount != 1:
            con.rollback()
            return False, "This token has already been used."

        #store the encrypted vote without student identity
        con.execute("""
            INSERT INTO votes (encrypted_vote, created_at)
            VALUES (?, ?)
        """, (
            encrypted_vote,
            datetime.now(timezone.utc).isoformat()
        ))

        #save both operations together
        con.commit()

        return True, "Vote submitted successfully."

    except Exception:
        con.rollback()
        return False, "Vote could not be submitted."

    finally:
        con.close()


# create the required tables
initialize_tokens()
initialize_votes()



In [16]:

student_input = widgets.Text(description='Student ID:')
password_input = widgets.Password(description='Password:')
token_input = widgets.Password(description='Voting Token:')

login_button = widgets.Button(description='Log in', button_style='primary')
login_output = widgets.Output()

candidate_select = widgets.RadioButtons(
    options=[(name, key) for key, name in CANDIDATES.items()],
    description='Candidate:'
)

confirm_button = widgets.Button(
    description='Confirm selection',
    button_style='success'
)

selection_output = widgets.Output()

selection_panel = widgets.VBox([
    widgets.HTML('<h3>Select one candidate</h3>'),
    token_input,
    candidate_select,
    confirm_button,
    selection_output
])

selection_panel.layout.display = 'none'

state = {'student_id': None, 'pending_candidate': None}

# Backend integration point.
# Function signature: handler(student_id, token, candidate_id)
_vote_handler = None

def register_vote_handler(handler):
    global _vote_handler
    if not callable(handler):
        raise TypeError('handler must be callable')
    _vote_handler = handler


def submit_vote(student_id, token, candidate_id):
    if candidate_id not in CANDIDATES:
        return False, 'Invalid candidate.'

    if _vote_handler is None:
        return False, 'Voting backend is not connected.'

    try:
        success, message = _vote_handler(
            student_id, token, candidate_id
        )
        return bool(success), str(message)

    except Exception:
        return False, 'Vote could not be submitted. Please contact the administrator.'


def on_login(_):
    student_id = student_input.value.strip()

    ok, message = authenticate(
        student_id,
        password_input.value
    )

    state['student_id'] = student_id if ok else None
    state['pending_candidate'] = None

    confirm_button.disabled = False
    selection_panel.layout.display = 'none'

    with login_output:
        clear_output()
        print(message)

        if ok:
            token, token_message = issue_token(student_id)

            if token is not None:
                token_input.value = token
                print(token_message)
                print('Copy and save your token safely.')
            else:
                token_input.value = ''
                print(token_message)
                print('If you saved your previous token, enter it below.')

            selection_panel.layout.display = ''

    with selection_output:
        clear_output()


def on_confirm(_):
    with selection_output:
        clear_output()

        if not state['student_id']:
            print('Please log in first.')
            return

        token = token_input.value.strip()
        selected = candidate_select.value

        state['pending_candidate'] = selected

        success, message = submit_vote(
            state['student_id'],
            token,
            selected
        )

        print(message)

        if success:
            confirm_button.disabled = True


login_button.on_click(on_login)
confirm_button.on_click(on_confirm)


# Temporary demo connection.
# Replace demo_encrypt_vote with Student 3's real AES function.
register_vote_handler(
    lambda student_id, token, candidate_id:
        cast_vote(
            student_id,
            token,
            candidate_id,
            demo_encrypt_vote,
            VOTING_DEADLINE
        )
)


display(widgets.VBox([
    widgets.HTML('<h2>Secure Classroom Voting — Student Portal</h2>'),
    student_input,
    password_input,
    login_button,
    login_output,
    selection_panel
]))


In [19]:

assert authenticate('student01', 'DemoPass01!')[0] is True
assert authenticate('student01', 'wrong-password')[0] is False
assert authenticate('unknown', 'anything')[0] is False
assert authenticate('student03', 'DemoPass03!')[0] is False

assert len(CANDIDATES) == 3

with connection() as con:
    tables = [r[0] for r in con.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    )]

    columns = [r[1] for r in con.execute(
        'PRAGMA table_info(students)'
    )]

assert 'students' in tables
# Other tables may be added when teammates integrate their components.
assert 'password_hash' in columns and 'password_salt' in columns

print('PASS: valid login, invalid password, unknown student, ineligible student,')
print('candidate options, student-only database schema and hashed credentials.')

assert submit_vote('student01', 'invalid-token', 'B')[0] is False


# Prepare a fresh test student for token testing
with connection() as con:
    con.execute("DELETE FROM votes WHERE encrypted_vote LIKE 'DEMO_ONLY_%'")
    con.execute("DELETE FROM tokens WHERE student_id = ?", ('test_student',))
    con.execute("DELETE FROM students WHERE student_id = ?", ('test_student',))

add_demo_student('test_student', 'TestPass123!', True)


# Token generation test
test_token, message = issue_token('test_student')

assert test_token is not None
assert len(test_token) > 20


# Verify that only the hash is stored
with connection() as con:
    row = con.execute(
        "SELECT token_hash, is_used FROM tokens WHERE student_id=?",
        ('test_student',)
    ).fetchone()

assert row is not None
assert row[0] == hash_token(test_token)
assert row[0] != test_token
assert row[1] == 0


# Verify correct token
valid, message = verify_token('test_student', test_token)
assert valid is True


# Verify incorrect token
valid, message = verify_token('test_student', 'wrong-token')
assert valid is False

print('PASS: token generation, hashed storage, and token validation.')


# Test duplicate vote prevention

with connection() as con:
    initial_count = con.execute(
        "SELECT COUNT(*) FROM votes"
    ).fetchone()[0]


success, message = cast_vote(
    'test_student',
    test_token,
    'A',
    demo_encrypt_vote,
    VOTING_DEADLINE
)

print("Success:", success)
print("Message:", message)

assert success is True


# Try using the same token again
success, message = cast_vote(
    'test_student',
    test_token,
    'B',
    demo_encrypt_vote,
    VOTING_DEADLINE
)

assert success is False
assert message == "This token has already been used."


# Check that only one new vote was stored
with connection() as con:
    vote_count = con.execute(
        "SELECT COUNT(*) FROM votes"
    ).fetchone()[0]

assert vote_count == initial_count + 1

print('PASS: duplicate voting is prevented.')


PASS: valid login, invalid password, unknown student, ineligible student,
candidate options, student-only database schema and hashed credentials.
PASS: token generation, hashed storage, and token validation.
Success: False
Message: Voting is closed.


AssertionError: 